# Baseline Model
### Evidence Retrieval
- Use word2Vec to encode test claims and all evidences
- Compute cosine similarity between claims and each evidence
- Select top 5 evidences that have the highest similarity score for each claim

### Claim Classification
- Use Random Forest to predict claim label

In [14]:
import json
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models.doc2vec import Doc2Vec
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from gensim.models import KeyedVectors, Word2Vec


In [2]:
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
	"""Preprocesses text by lowercasing, tokenizing, removing stopwords and stemming."""
	tokens = word_tokenize(text.lower())
	filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum() and word not in stop_words]
	return ' '.join(filtered_tokens)
	
# Load JSON data
def load_data(filepath):
	with open(filepath, 'r') as file:
		data = json.load(file)
	return data

[nltk_data] Downloading package punkt to /Users/chenluyao/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
train_claims_data = load_data('data/train-claims.json')
evidence_data = load_data('data/evidence.json')
dev_claims_data = load_data('data/dev-claims.json')
evidence_map = load_data('data/curated/preprocessed_evidence_map.json')
test_claims_data = load_data('data/test-claims-unlabelled.json')

In [4]:
evidence_df = pd.DataFrame(evidence_map.items(), columns=['id', 'evidence'])
evidence_df

,id,evidence
0,evidence-0,john bennet law english entrepreneur agricultu...
1,evidence-1,lindberg began profess career age eventu move ...
2,evidence-2,boston ladi cambridg vampir weekend
3,evidence-3,gerald franci goyer born octob profess ice hoc...
4,evidence-4,detect abnorm oxytocinerg function schizoaffec...
...,...,...
1208822,evidence-1208822,also properti contribut garag apart
1208823,evidence-1208823,class fn org fyrd volda
1208824,evidence-1208824,dragon storm game game collect card game
1208825,evidence-1208825,state zeriuani great realm tradit relat tribe ...


In [5]:
data_for_dataframe = []
for claim_id, claim_details in train_claims_data.items():
    claim_text = claim_details['claim_text']
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    data_for_dataframe.append({
            'claim': claim_text,
            'evidence': eids,
            'label': claim_label
        })

# Create DataFrame
train_claims_df = pd.DataFrame(data_for_dataframe)

train_claims_df['evidence_texts'] = train_claims_df['evidence'].apply(
    lambda x: [evidence_map[evidence_id] for evidence_id in x]
)

train_claims_df

,claim,evidence,label,evidence_texts
0,Not only is there no scientific evidence that ...,"[evidence-442946, evidence-1194317, evidence-1...",DISPUTED,[high concentr time atmosph concentr greater c...
1,El Niño drove record highs in global temperatu...,"[evidence-338219, evidence-1127398]",REFUTES,[climat chang due natur forc human activ subst...
2,"In 1946, PDO switched to a cool phase.","[evidence-530063, evidence-984887]",SUPPORTS,[evid rever prevail polar mean chang cool surf...
3,Weather Channel co-founder John Coleman provid...,"[evidence-1177431, evidence-782448, evidence-5...",DISPUTED,[convinc scientif evid human relea carbon diox...
4,"""January 2008 capped a 12 month period of glob...","[evidence-1010750, evidence-91661, evidence-72...",NOT_ENOUGH_INFO,"[averag temperatur, iranian persian calendar c..."
...,...,...,...,...
1223,Climate scientists say that aspects of the cas...,"[evidence-1055682, evidence-1047356, evidence-...",SUPPORTS,[fact climat chang made hurrican harvey deadli...
1224,"In its 5th assessment report in 2013, the IPCC...",[evidence-916755],SUPPORTS,[scientif consensu updat state ipcc fifth asse...
1225,"Since the mid 1970s, global temperatures have ...","[evidence-403673, evidence-889933, evidence-11...",NOT_ENOUGH_INFO,"[global warm, multipl independ produc instrume..."
1226,But abnormal temperature spikes in February an...,"[evidence-97375, evidence-562427, evidence-521...",NOT_ENOUGH_INFO,[lower air temperatur record may influenc grou...


In [6]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = claim_details['claim_text']
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    data_for_dataframe.append({
            'claim': claim_text,
            'evidence': eids,
            'label': claim_label
        })

# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)

dev_claims_df['evidence_texts'] = dev_claims_df['evidence'].apply(
    lambda x: [evidence_map[evidence_id] for evidence_id in x]
)

dev_claims_df

,claim,evidence,label,evidence_texts
0,[South Australia] has the most expensive elect...,"[evidence-67732, evidence-572512]",SUPPORTS,[citat need south australia highest retail pri...
1,when 3 per cent of total annual global emissio...,"[evidence-996421, evidence-1080858, evidence-2...",NOT_ENOUGH_INFO,[unep green economi report state agricultur op...
2,This means that the world is now 1C warmer tha...,"[evidence-889933, evidence-694262]",SUPPORTS,[multipl independ produc instrument dataset co...
3,"“As it happens, Zika may also be a good model ...","[evidence-422399, evidence-702226, evidence-28...",NOT_ENOUGH_INFO,[genet disord result deleteri mutat due sponta...
4,Greenland has only lost a tiny fraction of its...,"[evidence-52981, evidence-264761, evidence-947...",REFUTES,[iceberg calv happen averag greenland lost gt ...
...,...,...,...,...
149,"'To suddenly label CO2 as a ""pollutant"" is a d...","[evidence-409365, evidence-127519, evidence-85...",REFUTES,[state articl convent requir greenhou ga ghg c...
150,"after a natural orbitally driven warming, atmo...","[evidence-368192, evidence-261690, evidence-20...",NOT_ENOUGH_INFO,[increa atmosph concentr co greenhou gase meth...
151,Many of the world’s coral reefs are already ba...,"[evidence-1124018, evidence-995813, evidence-1...",NOT_ENOUGH_INFO,[tropic water contain nutrient yet coral reef ...
152,A recent study led by Lawrence Livermore Natio...,[evidence-660755],REFUTES,[studi david douglass cowork conclud commonli ...


In [11]:
all_texts = list(set(train_claims_df['claim'].tolist())) + list(set(evidence_map.values()))
processed_sentences = [sent.split() for sent in all_texts]

model = Word2Vec(
	sentences=processed_sentences,
	vector_size=300
)
model.save("word2vec.model")
word_vectors = model.wv
word_vectors.save("word2vec.wordvectors")

word_vectors = KeyedVectors.load('word2vec.wordvectors', mmap='r')
word_vectors['john'].shape

(300,)

In [17]:
evidence_df["vector"] = ""
for i in range(evidence_df.shape[0]):
    num_words = len(evidence_df["evidence"][i])
    vec = np.zeros((300,))
    if num_words > 0:
        for word in evidence_df["evidence"][i].split():
            if word in word_vectors:
                vec += word_vectors[word]
        vec = np.divide(vec, num_words)
    evidence_df["vector"][i] = vec
evidence_df

,id,evidence,vector
0,evidence-0,john bennet law english entrepreneur agricultu...,"[0.02377577129293952, -0.03124005438988669, 0...."
1,evidence-1,lindberg began profess career age eventu move ...,"[0.02472236152675192, -0.10207775023655366, 0...."
2,evidence-2,boston ladi cambridg vampir weekend,"[-0.019487700292042325, -0.05905972080571311, ..."
3,evidence-3,gerald franci goyer born octob profess ice hoc...,"[-0.10248949915863746, -0.06870882767577503, 0..."
4,evidence-4,detect abnorm oxytocinerg function schizoaffec...,"[0.031184247057271106, 0.02771218523213809, 0...."
...,...,...,...
1208822,evidence-1208822,also properti contribut garag apart,"[0.00021295632634844099, -0.008803912731153624..."
1208823,evidence-1208823,class fn org fyrd volda,"[0.01358989448003147, -0.010734255871047144, -..."
1208824,evidence-1208824,dragon storm game game collect card game,"[-0.06751279076561331, 0.016891338163986802, 0..."
1208825,evidence-1208825,state zeriuani great realm tradit relat tribe ...,"[0.033628723208318674, 0.02615595988014288, 0...."


In [18]:
data_for_dataframe = []
for claim_id, claim_details in test_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim': claim_text,
            'claim_text_raw': claim_details['claim_text']
        })
    
# Create DataFrame
test_claims_df = pd.DataFrame(data_for_dataframe)
test_claims_df 

,claim_id,claim,claim_text_raw
0,claim-2967,contribut wast heat global climat,The contribution of waste heat to the global c...
1,claim-979,warm weather worsen recent drought includ drie...,“Warm weather worsened the most recent five-ye...
2,claim-1609,greenland lost tini fraction ice mass,Greenland has only lost a tiny fraction of its...
3,claim-1020,global reef crisi necessarili mean extinct cor...,“The global reef crisis does not necessarily m...
4,claim-2599,small amount activ substanc caus larg effect,Small amounts of very active substances can ca...
...,...,...,...
148,claim-293,measur equip get old need replac often requir,When the measuring equipment gets old and need...
149,claim-910,cement iron steel petroleum refin industri cou...,"The cement, iron and steel, and petroleum refi..."
150,claim-2815,new studi surfac warm solar cycl found time hi...,A new peer-reviewed study on Surface Warming a...
151,claim-1652,strong co2 effect observ mani differ measur,The strong CO2 effect has been observed by man...


In [20]:
test_claims_df["vector"] = ""
for i in range(test_claims_df.shape[0]):
    num_words = len(test_claims_df["claim"][i])
    vec = np.zeros((300,))
    if num_words > 0:
        for word in test_claims_df["claim"][i].split():
            if word in word_vectors:
                vec += word_vectors[word]
        vec = np.divide(vec, num_words)
    test_claims_df["vector"][i] = vec
test_claims_df 

,claim_id,claim,claim_text_raw,vector
0,claim-2967,contribut wast heat global climat,The contribution of waste heat to the global c...,"[0.04300867771786271, 0.03020111206128742, 0.0..."
1,claim-979,warm weather worsen recent drought includ drie...,“Warm weather worsened the most recent five-ye...,"[0.019386866820209167, -0.00517881720819894, 0..."
2,claim-1609,greenland lost tini fraction ice mass,Greenland has only lost a tiny fraction of its...,"[0.008364970619614059, 0.09517915506620665, 0...."
3,claim-1020,global reef crisi necessarili mean extinct cor...,“The global reef crisis does not necessarily m...,"[0.03811211663263815, 0.04378649885593741, 0.0..."
4,claim-2599,small amount activ substanc caus larg effect,Small amounts of very active substances can ca...,"[0.04524045081978494, 0.029108072038401257, 0...."
...,...,...,...,...
148,claim-293,measur equip get old need replac often requir,When the measuring equipment gets old and need...,"[-0.08075365080601639, -0.0254601475265291, 0...."
149,claim-910,cement iron steel petroleum refin industri cou...,"The cement, iron and steel, and petroleum refi...","[-0.043651169510903184, 0.07096763277127419, 0..."
150,claim-2815,new studi surfac warm solar cycl found time hi...,A new peer-reviewed study on Surface Warming a...,"[0.023561748915014013, 0.11614103648251137, 0...."
151,claim-1652,strong co2 effect observ mani differ measur,The strong CO2 effect has been observed by man...,"[0.0747425376849119, 0.029363905621129414, 0.0..."


In [21]:
X = np.array(test_claims_df['vector'].values.tolist())
y = np.array(evidence_df['vector'].values.tolist())
sim = cosine_similarity(X, y)
print(sim.shape)
    

(153, 1208827)


In [24]:
# get top 5 evidence with highest similarity score with the claim
data = np.zeros((sim.shape[0], 5))
top_evidence_id = []
for i in range(sim.shape[0]):
	data[i] = np.argpartition(sim[i], -5)[-5:]
	top_evidence_id.append([evidence_df.iloc[int(ind)]['id'] for ind in data[i]])

test_claims_df['top5_evidence_id'] = top_evidence_id
test_claims_df = test_claims_df[["claim_id", "claim_text_raw", "top5_evidence_id"]]

# get texts of top 5 evidence
test_claims_df['evidence_texts'] = test_claims_df['top5_evidence_id'].apply(
    lambda x: [evidence_map[evidence_id] for evidence_id in x]
)

test_claims_df.to_csv("data/curated/test_evidence_retrieval.csv", index=False)
test_claims_df

/var/folders/df/4qk5nt6555bggnc39502n5b80000gn/T/ipykernel_49415/2024250276.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_claims_df['top5_evidence_id'] = top_evidence_id


,claim_id,claim_text_raw,top5_evidence_id,evidence_texts
0,claim-2967,The contribution of waste heat to the global c...,"[evidence-231959, evidence-953766, evidence-55...",[accord intergovern panel climat chang methan ...
1,claim-979,“Warm weather worsened the most recent five-ye...,"[evidence-178433, evidence-972542, evidence-24...",[anoth form sever weather drought prolong peri...
2,claim-1609,Greenland has only lost a tiny fraction of its...,"[evidence-399454, evidence-691825, evidence-12...",[increa rate ice mass loss greenland antarct i...
3,claim-1020,“The global reef crisis does not necessarily m...,"[evidence-450229, evidence-1103193, evidence-8...",[reason extinct believ habitat loss pollut int...
4,claim-2599,Small amounts of very active substances can ca...,"[evidence-772234, evidence-60866, evidence-909...","[sequenceom effect larg databa, tansymustard t..."
...,...,...,...,...
148,claim-293,When the measuring equipment gets old and need...,"[evidence-602437, evidence-1120180, evidence-3...",[tool primarili use anywh accur torqu requir n...
149,claim-910,"The cement, iron and steel, and petroleum refi...","[evidence-431016, evidence-838875, evidence-19...",[electron product miner fuel machineri transpo...
150,claim-2815,A new peer-reviewed study on Surface Warming a...,"[evidence-553725, evidence-431058, evidence-22...",[pattern solar irradi solar variat main driver...
151,claim-1652,The strong CO2 effect has been observed by man...,"[evidence-440714, evidence-276829, evidence-13...","[accur measur differ biodiv difficult, result ..."


### Claim Classification

In [55]:
# dev_claims_df = pd.read_csv('data/curated/dev_evidence_retrieval.csv', converters={
#     'top5_evidence_id': lambda x: np.array(x.strip("[]").split(','), dtype='int')})
# dev_claims_df

In [25]:
# combine claim text and evidence texts
X_train = train_claims_df['claim'] + train_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))
y_train = train_claims_df['label']

X_dev = dev_claims_df['claim'] + dev_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))
y_dev = dev_claims_df['label']

X_test = test_claims_df['claim_text_raw'] + test_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))

count_vectorizer = CountVectorizer()
X_train_count = count_vectorizer.fit_transform(X_train)
X_dev_count = count_vectorizer.transform(X_dev)
X_test_count = count_vectorizer.transform(X_test)

In [26]:
# Hyperparameters
n_estimators_values = [50, 100, 200]
max_depth_values = [None, 10, 20]

accuracy_scores_rf = []
for n_estimators in n_estimators_values:
    for max_depth in max_depth_values:
        rf_classifier = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        rf_classifier.fit(X_train_count, y_train)
        y_pred_rf = rf_classifier.predict(X_dev_count)
        
        accuracy_rf = accuracy_score(y_dev, y_pred_rf)
        accuracy_scores_rf.append(((n_estimators, max_depth), accuracy_rf))
        print(f"n_estimators = {n_estimators}, max_depth = {max_depth}: Accuracy = {accuracy_rf}")

print("Accuracy scores for Random Forest:")
for params, accuracy in accuracy_scores_rf:
    print(f"Parameters: {params}, Accuracy: {accuracy}")

n_estimators = 50, max_depth = None: Accuracy = 0.551948051948052
n_estimators = 50, max_depth = 10: Accuracy = 0.5194805194805194
n_estimators = 50, max_depth = 20: Accuracy = 0.5194805194805194
n_estimators = 100, max_depth = None: Accuracy = 0.5584415584415584
n_estimators = 100, max_depth = 10: Accuracy = 0.4935064935064935
n_estimators = 100, max_depth = 20: Accuracy = 0.525974025974026
n_estimators = 200, max_depth = None: Accuracy = 0.5584415584415584
n_estimators = 200, max_depth = 10: Accuracy = 0.5194805194805194
n_estimators = 200, max_depth = 20: Accuracy = 0.538961038961039
Accuracy scores for Random Forest:
Parameters: (50, None), Accuracy: 0.551948051948052
Parameters: (50, 10), Accuracy: 0.5194805194805194
Parameters: (50, 20), Accuracy: 0.5194805194805194
Parameters: (100, None), Accuracy: 0.5584415584415584
Parameters: (100, 10), Accuracy: 0.4935064935064935
Parameters: (100, 20), Accuracy: 0.525974025974026
Parameters: (200, None), Accuracy: 0.5584415584415584
Parame

In [27]:
# Apply to Test Set
rf_classifier = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42)
rf_classifier.fit(X_train_count, y_train)
y_pred = rf_classifier.predict(X_test_count)
test_claims_df["label"] = y_pred
test_claims_df['evidences'] = test_claims_df['top5_evidence_id'].apply(
    lambda x: ["evidence-" + str(evidence_id) for evidence_id in x]
)
test_claims_df.drop(columns=['evidence_texts', 'top5_evidence_id'], inplace=True)
test_claims_df.rename(columns={"claim_text_raw": "claim_text", "label": "claim_label"}, inplace=True)
test_claims_df.set_index('claim_id', inplace=True)
test_claims_df

,claim_text,claim_label,evidences
claim_id,,,
claim-2967,The contribution of waste heat to the global c...,SUPPORTS,"[evidence-evidence-231959, evidence-evidence-9..."
claim-979,“Warm weather worsened the most recent five-ye...,NOT_ENOUGH_INFO,"[evidence-evidence-178433, evidence-evidence-9..."
claim-1609,Greenland has only lost a tiny fraction of its...,SUPPORTS,"[evidence-evidence-399454, evidence-evidence-6..."
claim-1020,“The global reef crisis does not necessarily m...,SUPPORTS,"[evidence-evidence-450229, evidence-evidence-1..."
claim-2599,Small amounts of very active substances can ca...,SUPPORTS,"[evidence-evidence-772234, evidence-evidence-6..."
...,...,...,...
claim-293,When the measuring equipment gets old and need...,NOT_ENOUGH_INFO,"[evidence-evidence-602437, evidence-evidence-1..."
claim-910,"The cement, iron and steel, and petroleum refi...",SUPPORTS,"[evidence-evidence-431016, evidence-evidence-8..."
claim-2815,A new peer-reviewed study on Surface Warming a...,SUPPORTS,"[evidence-evidence-553725, evidence-evidence-4..."


In [28]:
# convert to json file
from json import loads
result = test_claims_df.to_json(orient="index")
with open('data/curated/test-output.json', 'w') as f:
    f.write(result)